## RFM Feature Engineering


RFM is the most widely used customer feature set in retail ML:
- **Recency (R):** Days since the customer's last purchase.
  Lower = more engaged.
- **Frequency (F):** Total number of distinct orders placed.
  Higher = more loyal.
- **Monetary (M):** Total revenue generated by the customer.
  Higher = more valuable.


These three numbers compress an entire transaction history into a
compact, interpretable representation used by every major retailer.
They are the input features for both CLV prediction and segmentation.


In [ ]:
import os, zipfile
import urllib.request
import pandas as pd

url = ('https://archive.ics.uci.edu/static/public/352/online+retail.zip')
raw_data_path = r'C:\Git projects\customer_lifetime_value\data\data_raw\online_retail.zip'
#os.makedirs(os.path.dirname(save_path), exist_ok=True) # Ensure the directory exists
urllib.request.urlretrieve(url, raw_data_path)

file_list = zipfile.ZipFile(raw_data_path)
print (file_list.namelist())

['Online Retail.xlsx']


In [ ]:
# extract a specific file from the zip container
f = file_list.open("Online Retail.xlsx")

# save the extraced file 
content = f.read()
f = open('C:\\Git projects\\customer_lifetime_value\\data\\data_raw\\Online Retail.xlsx', 'wb')
f.write(content)
f.close()

In [23]:
# The file is an Excel file
df = pd.read_excel(r'C:\Git projects\customer_lifetime_value\data\data_raw\Online Retail.xlsx', dtype={'CustomerID': str})
print(df.shape)       # Expected: ~541,909 rows
print(df.dtypes)
df.head()

(541909, 8)
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID             object
Country                object
dtype: object


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


In [24]:
# Drop rows with missing CustomerID — cannot build RFM without it
df = df.dropna(subset=['CustomerID'])


# Remove cancellations (InvoiceNo starting with 'C')
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]


# Remove rows with non-positive Quantity or UnitPrice
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]


# Compute line-level revenue
df['Revenue'] = df['Quantity'] * df['UnitPrice']


# Parse InvoiceDate
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])


print(f'Clean dataset: {df.shape}')
print(f'Unique customers: {df["CustomerID"].nunique()}')


Clean dataset: (397884, 9)
Unique customers: 4338


In [25]:
# Snapshot date: one day after the last transaction
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)


rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('Revenue',     'sum')
).reset_index()


print(rfm.describe())
rfm.head(10)


           Recency    Frequency       Monetary
count  4338.000000  4338.000000    4338.000000
mean     92.536422     4.272015    2054.266460
std     100.014169     7.697998    8989.230441
min       1.000000     1.000000       3.750000
25%      18.000000     1.000000     307.415000
50%      51.000000     2.000000     674.485000
75%     142.000000     5.000000    1661.740000
max     374.000000   209.000000  280206.020000


,CustomerID,Recency,Frequency,Monetary
0,12346,326,1,77183.60
1,12347,2,7,4310.00
2,12348,75,4,1797.24
3,12349,19,1,1757.55
4,12350,310,1,334.40
5,12352,36,8,2506.04
6,12353,204,1,89.00
7,12354,232,1,1079.40
8,12355,214,1,459.40
9,12356,23,3,2811.43


In [26]:
# Define windows
obs_end    = pd.Timestamp('2011-09-30')
target_end = pd.Timestamp('2011-12-09')   # dataset end


df_obs    = df[df['InvoiceDate'] <= obs_end]
df_target = df[(df['InvoiceDate'] > obs_end) & (df['InvoiceDate'] <= target_end)]


# RFM on observation window
snap_obs = obs_end + pd.Timedelta(days=1)
rfm_obs = df_obs.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (snap_obs - x.max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('Revenue',     'sum')
).reset_index()


# Target: 90-day forward spend
clv_target = df_target.groupby('CustomerID')['Revenue'].sum().reset_index()
clv_target.columns = ['CustomerID', 'CLV_90d']


# Merge — customers with no future spend get 0
rfm_clv = rfm_obs.merge(clv_target, on='CustomerID', how='left')
rfm_clv['CLV_90d'] = rfm_clv['CLV_90d'].fillna(0)


print(f'Customers with future spend > 0: {(rfm_clv["CLV_90d"]>0).sum()}')
rfm_clv.to_csv('C:\\Git projects\\customer_lifetime_value\\data\\data_processed\\rfm_clv.csv', index=False)
print('Saved: rfm_clv.csv')


Customers with future spend > 0: 1843
Saved: rfm_clv.csv


## RFM Summary Interpretation


**Recency distribution:** Most customers have high recency (many days since
last purchase), which is typical in non-subscription retail. A long tail of
recently active customers represents the most actionable segment.


**Frequency:** The majority of customers placed only 1-2 orders.
Multi-order customers are a small but disproportionately valuable group.


**Monetary:** Right-skewed distribution — a small number of customers
generate most revenue. This is the classic Pareto 80/20 pattern in retail.


**CLV target:** ~18% of customers made at least one purchase in the
90-day target window. The rest are churned or lapsed — a modeling
challenge addressed in Notebook 2.
